# 310 · Four frequencies on one output

**Level:** advanced · **Topics:** Gen2, SHFSG, output-router, calibration, device-setup · **Type:** fix the bugs

A customer has an SHFSG8 with the Output Router and Adder option (RTR).
They want to play **four tones at the same time on output 1** to drive one qubit:

| Signal | Sequencer (port) | Tone |
| --- | --- | --- |
| `q0/drive_f0` | `SGCHANNELS/0` | 4.0 GHz − 110 MHz |
| `q0_add/drive_f1` | `SGCHANNELS/2` | 4.0 GHz − 58 MHz |
| `q0_add/drive_f2` | `SGCHANNELS/3` | 4.0 GHz + 50 MHz |
| `q0_add/drive_f3` | `SGCHANNELS/4` | 4.0 GHz + 200 MHz |

Only output 1 (`SGCHANNELS/0`) is connected to the fridge. The three `q0_add` signals borrow the sequencers of unused channels, and the router adds them to output 1.

The notebook looks complete, but it has **four bugs**:

- three raise an error,
- one is silent (the evil one): everything runs, but the tones do not reach output 1.

Everything runs in emulation, no instrument needed.

In [ ]:
from laboneq.simple import *

## Device setup

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html) ·
[Instrument options](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/01_instrument_options.html)

In [ ]:
setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(
    SHFSG(uid="shfsg", address="DEV12500", device_options="SHFSG8")
)
setup.add_connections(
    "shfsg",
    create_connection(to_signal="q0/drive_f0", ports="SGCHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f1", ports="SGCHANNELS/2/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f2", ports="SGCHANNELS/3/OUTPUT"),
    create_connection(to_signal="q0_add/drive_f3", ports="SGCHANNELS/4/OUTPUT"),
)

## Calibration

📖 [Calibrating signals](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/02_logical_signals/concepts/00_calibrating_signals.html) ·
[Output router tutorial](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/10_advanced_topics/tutorials/08_output_router.html) ·
[`OutputRoute` reference](https://docs.zhinst.com/labone_q_user_manual/core/reference/dsl/calibration.html)

In [ ]:
calibration = Calibration()

calibration["q0/drive_f0"] = SignalCalibration(
    oscillator=Oscillator(frequency=-110e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
)
calibration["q0_add/drive_f1"] = SignalCalibration(
    oscillator=Oscillator(frequency=-58e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.0e9),
    range=10,
    added_output=OutputRoute(source="q0/drive_f0", amplitude_scaling=0.2, phase_shift=0.0),
)
calibration["q0_add/drive_f2"] = SignalCalibration(
    oscillator=Oscillator(frequency=50e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.6e9),
    range=10,
    added_output=OutputRoute(source="q0/drive_f0", amplitude_scaling=0.2, phase_shift=0.0),
)
calibration["q0_add/drive_f3"] = SignalCalibration(
    oscillator=Oscillator(frequency=200e6, modulation_type=ModulationType.HARDWARE),
    local_oscillator=Oscillator(frequency=4.2e9),
    range=10,
    added_output=OutputRoute(source="q0/drive_f0", amplitude_scaling=0.2, phase_shift=0.0),
)

setup.set_calibration(calibration)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## Experiment

All four tones play together, with the same envelope.

📖 [Sections and pulses](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/index.html) ·
[Pulse amplitude and phase](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/concepts/06_pulses_amplitude_and_phase.html)

In [ ]:
drive_pulse = pulse_library.const(length=1e-6)


@dsl.experiment(signals=["f0", "f1", "f2", "f3"])
def four_tones():
    with dsl.acquire_loop_rt(count=16):
        with dsl.section(name="drive"):
            dsl.play("f0", drive_pulse, amplitude=0.2)
            dsl.play("f1", drive_pulse, amplitude=0.8)
            dsl.play("f2", drive_pulse, amplitude=0.8)
            dsl.play("f3", drive_pulse, amplitude=0.8)
        with dsl.section(name="wait"):
            dsl.delay("f0", time=1e-6)


experiment = four_tones()
experiment.set_signal_map(
    {
        "f0": "q0/drive_f0",
        "f1": "q0_add/drive_f1",
        "f2": "q0_add/drive_f2",
        "f3": "q0_add/drive_f3",
    }
)

## Compile and run

📖 [Experiment](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/index.html)

In [ ]:
compiled = session.compile(experiment)
results = session.run(compiled)

## Check the routes

The output simulator shows each sequencer on its own, not the sum made by the router.
So we check the routes in the calibration instead: for each signal, which other signals are added to its output?

In [ ]:
# check: "target <- sources added to it"
for name in ["q0/drive_f0", "q0_add/drive_f1", "q0_add/drive_f2", "q0_add/drive_f3"]:
    routes = setup.logical_signal_by_uid(name).calibration.added_outputs
    sources = []
    if routes is not None:
        for route in routes:
            sources.append(route.source)
    print(name, "<-", sources)

**Expected result**

```
q0/drive_f0 <- ['q0_add/drive_f1', 'q0_add/drive_f2', 'q0_add/drive_f3']
q0_add/drive_f1 <- []
q0_add/drive_f2 <- []
q0_add/drive_f3 <- []
```

## Questions

1. Fix the three errors, one at a time. Read each error message: which one tells you exactly what to do?
2. Find the silent bug with the check above and fix it.
3. With the routes fixed: what is the amplitude of each tone at output 1? Change the experiment so that every tone has amplitude 0.2 (0.8 in total).
4. Why not simply connect all four signals to `SGCHANNELS/0/OUTPUT`? Try it.
5. The customer downgrades to an SHFSG4 with RTR. Which ports can the `q0_add` signals use?

<details>
<summary>Hint 1</summary>

The keyword is `added_outputs` and it takes a list.
</details>

<details>
<summary>Hint 2</summary>

On the SHFSG, two neighbouring channels share one synthesizer (1–2, 3–4, 5–6, 7–8).
</details>

<details>
<summary>Hint 3</summary>

Read the description of `source` in the `OutputRoute` reference: is the route defined on the signal that gives, or on the signal that receives?
</details>

The solution is in [solution.ipynb](solution.ipynb).